In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.functions import (col, min, max, count, date_trunc, md5,
                                   datediff, when, current_timestamp)
from datetime import datetime

# Widget with default value
dbutils.widgets.text("table_name", "dim_customers")
table_name = dbutils.widgets.get("table_name")

# Configuration & Paths
silver_base = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/silver/"
gold_base = "abfss://curated@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/gold/"
target_table = table_name
gold_path = f"{gold_base}{target_table}"

# Silver Sources
df_orders = spark.read.format("delta").load(f"{silver_base}olist_orders_dataset")
df_cust = spark.read.format("delta").load(f"{silver_base}olist_customers_dataset")
df_geo = spark.read.format("delta").load(f"{silver_base}olist_geolocation_dataset")

# Transformation Logic
df_cust_orders = (df_orders.alias("o")
                  .join(df_cust.alias("c"), "customer_id", "left")
                  .groupBy("customer_unique_id")
                  .agg(
                      F.min("order_purchase_timestamp").alias("first_order_date"),
                      F.max("order_purchase_timestamp").alias("last_order_date"),
                      F.count("order_id").alias("total_orders")))

df_dim_base = (df_cust.alias("c")
               .join(df_cust_orders.alias("co"), "customer_unique_id", "left")
               .join(df_geo.alias("g"), F.col("c.customer_zip_code_prefix") == F.col("g.geolocation_zip_code_prefix"), "left")
           .select(
              F.md5(F.col("c.customer_id")).cast("string").alias("customer_key"),
              "c.customer_id",
              "c.customer_unique_id",
              "c.customer_city",
              "c.customer_state",
              "c.customer_zip_code_prefix",
              F.col("g.geolocation_lat").alias("latitude"),
              F.col("g.geolocation_lng").alias("longitude"),
              F.col("co.first_order_date").cast("date"),
              F.col("co.last_order_date").cast("date"),
              F.col("co.total_orders").cast("int"),
              F.when(F.col("co.total_orders") > 1, 1).otherwise(0).alias("is_repeat_customer"),
              F.datediff("co.last_order_date", "co.first_order_date").alias("customer_tenure_days"),
              F.date_trunc("second", F.current_timestamp()).alias("gold_load_at")
          ))

# Unknown record
now = datetime.now().replace(microsecond=0)
unknown_data = [(-1, "UNKNOWN", "UNKNOWN", "NA", "NA", None, None, None, None, None, 0, 0, 0, now)]
target_schema = df_dim_base.schema
df_unknown = spark.createDataFrame(unknown_data, schema=target_schema)
df_final = df_dim_base.unionByName(df_unknown)

# Incremental Merge (Upsert)
print(f"--> Starting Gold Load: {target_table}")

if DeltaTable.isDeltaTable(spark, gold_path):
    dt_gold = DeltaTable.forPath(spark, gold_path)

    (dt_gold.alias("target")
     .merge(df_final.alias("source"), "target.customer_id = source.customer_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"--> [Merge] {target_table} updated.")
else:
    #First time load
    df_final.write.format("delta").mode("overwrite").save(gold_path)
    print(f"--> [INIT] {target_table} created.")

# Audit & Exit
dt_final = DeltaTable.forPath(spark, gold_path)

last_operation = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_inserted = int(last_operation.get("numTargetRowsInserted", 0))
rows_updated = int(last_operation.get("numTargetRowsUpdated", 0))
total_rows = last_operation.get("numOutputRows", "Check History")

print("-" * 50)
print(f"--> Table: {table_name} | Status: Success")
print(f"--> Rows Processed in last run: {rows_inserted + rows_updated}")
print("-" * 50)

dbutils.notebook.exit(f"Success | Inserted: {rows_inserted} | Updated: {rows_updated}")